# Day 25 — Is the volatility on the right scale?

**Acceptance:** annualized vol for a large cap lands in a plausible 15–35% band, not 3% or 300%.

Realized volatility at session *t* over a window of *w* sessions is the sample standard
deviation of the *w* daily log returns ending at *t*, scaled to a year:

$$\sigma_w(t) = \sqrt{252}\;\operatorname{std}\big(\ln p_{t-w+1} - \ln p_{t-w},\;\dots,\;\ln p_t - \ln p_{t-1}\big)$$

The two classic mistakes both show up as an order of magnitude: forgetting the
$\sqrt{252}$ gives ~1–2% (a *daily* number), and scaling by 252 instead of its square
root gives ~300%+. *p* is `adj_close_eur`, the series the returns are on.

The stock is **AAPL**, the largest cap in the universe. The hand calculation below
does not use `finmgr.features`.

In [1]:
import math

import exchange_calendars as xcals
import numpy as np
import pandas as pd

from finmgr.data.benchmarks import read_total_return

TICKER = "AAPL"
AS_OF = "2026-10-14"  # pinned so the numbers below are reproducible
t = pd.Timestamp(AS_OF).date()

bars = read_total_return(TICKER, as_of=AS_OF, start="2024-06-01")
bars = bars.sort_values("date").reset_index(drop=True)
print(len(bars), "bars,", bars["date"].min(), "to", bars["date"].max())

594 bars, 2024-06-03 to 2026-10-14


## Step 1 — the rows are sessions

The 252-session window needs 253 prices. Check they are exactly XNAS's sessions, so a
row is a session and nothing is stretched.

In [2]:
i = bars.index[bars["date"] == t][0]
window = bars.loc[i - 252 : i]
xnas = [s.date() for s in xcals.get_calendar("XNAS").sessions_in_range(window["date"].iloc[0], t)]
assert list(window["date"]) == xnas, "store and calendar disagree"
print(len(window), "prices,", len(xnas), "XNAS sessions, from", xnas[0], "to", xnas[-1])

253 prices, 253 XNAS sessions, from 2025-10-13 to 2026-10-14


## Step 2 — the hand calculation

In [3]:
logp = np.log(bars["adj_close_eur"].to_numpy())
r = np.diff(logp)[:i]  # r[k] is the return from row k to row k+1; the last is into t
hand = {w: float(np.std(r[-w:], ddof=1) * math.sqrt(252)) for w in (21, 63, 252)}
daily = float(np.std(r[-252:], ddof=1))
print(f"daily std of log returns, last 252 sessions: {daily:.4%}")
for w, v in hand.items():
    print(f"vol_{w}d by hand: {v:.4%}")
print(
    f"for contrast — no annualization: {daily:.2%}; scaled by 252 not its root: {daily * 252:.0%}"
)

daily std of log returns, last 252 sessions: 1.5992%
vol_21d by hand: 17.9677%
vol_63d by hand: 27.6298%
vol_252d by hand: 25.3859%
for contrast — no annualization: 1.60%; scaled by 252 not its root: 403%


## Step 3 — the module, and the comparison

In [4]:
from finmgr.features.volatility import VOLATILITY_COLUMNS, read_volatility

mod = read_volatility(TICKER, as_of=AS_OF, start=AS_OF).set_index("date").loc[t]
for w in (21, 63, 252):
    print(f"vol_{w}d  hand {hand[w]:.6%}   module {mod[f'vol_{w}d']:.6%}")
    assert math.isclose(mod[f"vol_{w}d"], hand[w], rel_tol=1e-10)
print("MATCH")
mod[list(VOLATILITY_COLUMNS)]

vol_21d  hand 17.967662%   module 17.967662%
vol_63d  hand 27.629767%   module 27.629767%
vol_252d  hand 25.385934%   module 25.385934%
MATCH


vol_21d        0.179677
vol_63d        0.276298
vol_252d       0.253859
downdev_63d    0.213414
atr_14         6.677249
atr_14_pct     0.019469
mdd_252d      -0.144138
Name: 2026-10-14, dtype: object

## Step 4 — the acceptance check

The one-year number, in euros and (for a cross-check without the currency leg) in
dollars, must sit in 15–35%.

In [5]:
usd = read_volatility(TICKER, as_of=AS_OF, start=AS_OF, price="adj_close").set_index("date").loc[t]
print(f"AAPL vol_252d in EUR: {mod['vol_252d']:.1%}   in USD: {usd['vol_252d']:.1%}")
assert 0.15 <= mod["vol_252d"] <= 0.35
assert 0.15 <= usd["vol_252d"] <= 0.35
print("IN BAND")

AAPL vol_252d in EUR: 25.4%   in USD: 24.5%
IN BAND


## Step 5 — not just one stock, not just one day

The other mega caps on the same date, and AAPL's `vol_252d` at every month-end of the
last three years. A single in-band number could be luck; a scale bug would show
everywhere.

In [6]:
MEGA = ["AAPL", "MSFT", "GOOGL", "AMZN", "JPM", "JNJ", "KO", "PG", "NESN.SW", "SAP.DE"]
panel = read_volatility(MEGA, as_of=AS_OF, start=AS_OF).set_index("ticker")
print(panel[["vol_21d", "vol_63d", "vol_252d", "downdev_63d", "atr_14_pct", "mdd_252d"]].round(3))
assert panel["vol_252d"].between(0.10, 0.45).all()
assert panel["vol_252d"].notna().all()
assert 0.15 <= panel["vol_252d"].median() <= 0.35

         vol_21d  vol_63d  vol_252d  downdev_63d  atr_14_pct  mdd_252d
ticker                                                                
AAPL       0.180    0.276     0.254        0.213       0.019    -0.144
AMZN       0.252    0.392     0.340        0.193       0.023    -0.240
GOOGL      0.228    0.326     0.318        0.241       0.024    -0.188
JNJ        0.260    0.237     0.212        0.185       0.024    -0.118
JPM        0.176    0.179     0.231        0.142       0.019    -0.142
KO         0.166    0.203     0.199        0.120       0.014    -0.080
MSFT       0.225    0.367     0.320        0.150       0.022    -0.332
NESN.SW    0.184    0.261     0.234        0.224       0.015    -0.152
PG         0.181    0.177     0.208        0.114       0.016    -0.147
SAP.DE     0.232    0.395     0.416        0.193       0.028    -0.461


In [7]:
hist = read_volatility(TICKER, as_of=AS_OF, start="2023-10-01").set_index("date")
month_ends = hist.groupby(pd.to_datetime(hist.index).to_period("M")).tail(1)
print(month_ends["vol_252d"].describe().round(3))
assert month_ends["vol_252d"].between(0.12, 0.45).all()
print(f"{len(month_ends)} month-ends, all between 12% and 45%")

count    37.000
mean      0.265
std       0.046
min       0.202
25%       0.231
50%       0.251
75%       0.323
max       0.337
Name: vol_252d, dtype: float64
37 month-ends, all between 12% and 45%


## Step 6 — the other risk columns, read side by side

Downside deviation should be below vol (only the falls count), ATR as a fraction of
price should be in the 1–4% a day a large cap shows, and the one-year drawdown is
non-positive.

In [8]:
assert (panel["downdev_63d"] < panel["vol_63d"]).all()
assert panel["atr_14_pct"].between(0.005, 0.05).all()
assert (panel["mdd_252d"] <= 0).all()
print("ok")

ok
